In [1]:
!pip install pandas plotly openpyxl

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
import csv

pio.renderers.default = "notebook"

In [5]:
pio.renderers.default = "browser"

In [7]:
def find_column(df, candidates):
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in normalized:
            return normalized[key]
    return None


def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [9]:
def detect_csv_separator(file_path):
    file_path = Path(file_path)

    with open(file_path, "r", encoding="utf-8-sig", errors="ignore") as f:
        sample = f.read(5000)

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t", "|"])
        return dialect.delimiter
    except Exception:
        # fallback
        if sample.count(";") > sample.count(","):
            return ";"
        return ","


def load_data(file_path):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        xls = pd.ExcelFile(file_path)
        first_sheet = xls.sheet_names[0]
        df = pd.read_excel(file_path, sheet_name=first_sheet)
        source_name = first_sheet

    elif suffix == ".csv":
        sep = detect_csv_separator(file_path)
        df = pd.read_csv(file_path, sep=sep)
        source_name = file_path.stem
        print(f"Detected CSV separator: {repr(sep)}")

    else:
        raise ValueError("Unsupported file format. Use .xlsx, .xls, or .csv")

    df.columns = [str(c).replace("Â°C", "°C").strip() for c in df.columns]
    return df, source_name

In [11]:
def clean_gps_data(
    df,
    lat_col,
    lon_col,
    timestamp_col=None,
    max_speed_kmh=120,
    max_jump_m=150,
    use_rolling_filter=True,
    rolling_window=5,
    rolling_threshold_deg=0.0015
):
    df = df.copy()

    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")

    df = df.dropna(subset=[lat_col, lon_col]).copy()
    df = df[df[lat_col].between(-90, 90)]
    df = df[df[lon_col].between(-180, 180)]
    df = df[~((df[lat_col] == 0) & (df[lon_col] == 0))].copy()

    if timestamp_col is not None:
        df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
        df = df.sort_values(timestamp_col).copy()
    else:
        df = df.reset_index(drop=True)

    df = df.loc[
        ~((df[lat_col].shift() == df[lat_col]) & (df[lon_col].shift() == df[lon_col]))
    ].copy()

    df["prev_lat"] = df[lat_col].shift(1)
    df["prev_lon"] = df[lon_col].shift(1)

    valid_prev = df["prev_lat"].notna() & df["prev_lon"].notna()
    df["jump_m"] = np.nan

    df.loc[valid_prev, "jump_m"] = haversine_m(
        df.loc[valid_prev, "prev_lat"],
        df.loc[valid_prev, "prev_lon"],
        df.loc[valid_prev, lat_col],
        df.loc[valid_prev, lon_col]
    )

    if timestamp_col is not None:
        df["dt_s"] = df[timestamp_col].diff().dt.total_seconds()
        df["speed_kmh_est"] = (df["jump_m"] / df["dt_s"]) * 3.6

        speed_mask = (
            df["speed_kmh_est"].isna() |
            (df["dt_s"] <= 0) |
            (df["speed_kmh_est"] <= max_speed_kmh)
        )
    else:
        speed_mask = pd.Series(True, index=df.index)

    jump_mask = df["jump_m"].isna() | (df["jump_m"] <= max_jump_m)
    df = df[speed_mask & jump_mask].copy()

    if use_rolling_filter and len(df) >= rolling_window:
        lat_med = df[lat_col].rolling(window=rolling_window, center=True, min_periods=1).median()
        lon_med = df[lon_col].rolling(window=rolling_window, center=True, min_periods=1).median()

        lat_dev = (df[lat_col] - lat_med).abs()
        lon_dev = (df[lon_col] - lon_med).abs()

        smooth_mask = (lat_dev <= rolling_threshold_deg) & (lon_dev <= rolling_threshold_deg)
        df = df[smooth_mask].copy()

    drop_cols = ["prev_lat", "prev_lon", "jump_m", "dt_s", "speed_kmh_est"]
    df = df.drop(columns=drop_cols, errors="ignore").reset_index(drop=True)

    return df

In [13]:
def build_route_map(
    file_path,
    output_html="route_map_hover_cleaned.html",
    map_zoom=15,
    marker_size=6,
    max_speed_kmh=120,
    max_jump_m=150
):
    df, source_name = load_data(file_path)

    timestamp_col = find_column(df, ["Timestamp", "Time", "DateTime", "Datetime"])
    lat_col = find_column(df, ["Latitude", "Lat", "GPS Latitude"])
    lon_col = find_column(df, ["Longitude", "Lon", "Lng", "GPS Longitude"])

    flir_col = find_column(df, [
        "FLIR Temp (°C)",
        "FLIR Temperature (°C)",
        "Surface Temp (°C)",
        "Surface Temperature (°C)"
    ])

    air_col = find_column(df, [
        "Ambient Temp (°C)",
        "Air Temp (°C)",
        "Air Temperature (°C)",
        "Temperature (°C)"
    ])

    hum_col = find_column(df, [
        "Humidity (%)",
        "Humidity (%RH)",
        "Relative Humidity (%)",
        "Relative Humidity (%RH)"
    ])

    speed_col = find_column(df, ["Speed (km/h)", "Speed"])
    alt_col = find_column(df, ["Altitude (m)", "Altitude"])
    sats_col = find_column(df, ["Satellites", "Satellite Count"])
    hdop_col = find_column(df, ["HDOP"])

    if lat_col is None or lon_col is None:
        raise ValueError("Latitude/Longitude columns were not found.")

    n_before = len(df)

    df = clean_gps_data(
        df=df,
        lat_col=lat_col,
        lon_col=lon_col,
        timestamp_col=timestamp_col,
        max_speed_kmh=max_speed_kmh,
        max_jump_m=max_jump_m,
        use_rolling_filter=True,
        rolling_window=5,
        rolling_threshold_deg=0.0015
    )

    n_after = len(df)

    if df.empty:
        raise ValueError("No valid GPS rows remained after filtering.")

    hover_text = []
    for _, row in df.iterrows():
        parts = [f"<b>Source:</b> {source_name}"]

        if timestamp_col is not None and pd.notna(row[timestamp_col]):
            parts.append(f"<b>Time:</b> {row[timestamp_col]}")

        if flir_col is not None and pd.notna(row[flir_col]):
            parts.append(f"<b>Surface Temp (FLIR):</b> {row[flir_col]:.2f} °C")

        if air_col is not None and pd.notna(row[air_col]):
            parts.append(f"<b>Air Temp:</b> {row[air_col]:.2f} °C")

        if hum_col is not None and pd.notna(row[hum_col]):
            parts.append(f"<b>Humidity:</b> {row[hum_col]:.2f} %RH")

        if speed_col is not None and pd.notna(row[speed_col]):
            parts.append(f"<b>Speed:</b> {row[speed_col]:.2f} km/h")

        if alt_col is not None and pd.notna(row[alt_col]):
            parts.append(f"<b>Altitude:</b> {row[alt_col]:.2f} m")

        if sats_col is not None and pd.notna(row[sats_col]):
            parts.append(f"<b>Satellites:</b> {row[sats_col]}")

        if hdop_col is not None and pd.notna(row[hdop_col]):
            parts.append(f"<b>HDOP:</b> {row[hdop_col]:.2f}")

        hover_text.append("<br>".join(parts))

    center_lat = df[lat_col].mean()
    center_lon = df[lon_col].mean()

    fig = go.Figure()

    fig.add_trace(
        go.Scattermapbox(
            lat=df[lat_col],
            lon=df[lon_col],
            mode="lines+markers",
            name="Route",
            text=hover_text,
            hoverinfo="text",
            line=dict(width=4),
            marker=dict(size=marker_size),
        )
    )

    fig.add_trace(
        go.Scattermapbox(
            lat=[df.iloc[0][lat_col]],
            lon=[df.iloc[0][lon_col]],
            mode="markers",
            name="Start",
            text=["Start Point"],
            hoverinfo="text",
            marker=dict(size=12, symbol="circle"),
        )
    )

    fig.add_trace(
        go.Scattermapbox(
            lat=[df.iloc[-1][lat_col]],
            lon=[df.iloc[-1][lon_col]],
            mode="markers",
            name="End",
            text=["End Point"],
            hoverinfo="text",
            marker=dict(size=12, symbol="circle"),
        )
    )

    fig.update_layout(
        title=f"Route Map (GPS cleaned: {n_before} → {n_after} points)",
        mapbox=dict(
            style="open-street-map",
            center=dict(lat=center_lat, lon=center_lon),
            zoom=map_zoom,
        ),
        margin=dict(l=10, r=10, t=50, b=10),
        hovermode="closest",
        legend=dict(x=0.01, y=0.99),
    )

    fig.write_html(
        output_html,
        include_plotlyjs=True,
        config={"scrollZoom": True, "displaylogo": False}
    )

    fig.show(config={"scrollZoom": True, "displaylogo": False})

    print(f"Map saved to: {output_html}")
    print(f"GPS points before filtering: {n_before}")
    print(f"GPS points after filtering:  {n_after}")

In [15]:
# Excel file
file_path = r"C:\Flir\Analysis.xlsx"

In [ ]:
#CSV file
file_path = r"C:\Flir\Analysis.csv"

In [19]:
build_route_map(
    file_path=file_path,
    output_html="route_map_hover.html",
    map_zoom=15,
    marker_size=5,
    max_speed_kmh=100,
    max_jump_m=120
)

Map saved to: route_map_hover.html
GPS points before filtering: 1015
GPS points after filtering:  1009
